# Heat Exchanger Trade-off Analysis: Manufacturing vs Operating Cost

This notebook analyzes the trade-off between manufacturing cost (mainly evaporator size/cost) and operating cost (COP/efficiency) for an 8 kW air-to-refrigerant heat pump with an active expander. We use TESPy for thermodynamic calculations and real-world evaporator price data.

In [34]:
# Custom subclass for detailed evaporator simulation
import sys
sys.path.append('../')
from HeatPumpStudy import HeatPumpStudy
from tespy.components import HeatExchanger, Drum, Valve, Source, Sink, Turbine, Compressor, Pump, CycleCloser
from tespy.connections import Connection
from CoolProp.CoolProp import PropsSI as PSI
import uuid

class EvaporatorHeatPumpStudy(HeatPumpStudy):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def setup_components_and_connections(self):
        expansion_type = Valve
        if self.expansion_device == "expander":
            expansion_type = Turbine
        elif self.expansion_device != "expansionValve":
            raise ValueError("expansion_device must be either 'expansionValve' or 'expander'")

        component_list = [
            ("evaporator", HeatExchanger),
            ("compressor", Compressor),
            ("condenser", HeatExchanger),
            ("expansionValve", Valve),
            ("cycle_closer", CycleCloser),
            ("condenser_coolant_source", Source),
            ("condenser_coolant_sink", Sink),
            ("evap_air_source", Source),
            ("evap_air_sink", Sink)
        ]
        if self.expansion_device == "expander":
            component_list.append(("expander", Turbine))

        connection_list = [
            ("cycle_closer", "out1", "evaporator", "in1"),
            ("evaporator", "out1", "compressor", "in1"),
            ("compressor", "out1", "condenser", "in1"),
            ("condenser", "out1", "expansionValve", "in1"),
            ("condenser_coolant_source", "out1", "condenser", "in2"),
            ("condenser", "out2", "condenser_coolant_sink", "in1"),
            ("evap_air_source", "out1", "evaporator", "in2"),
            ("evaporator", "out2", "evap_air_sink", "in1")
        ]
        if self.expansion_device == "expander":
            connection_list.append(("expansionValve", "out1", "expander", "in1"))
        connection_list.append((self.expansion_device, "out1", "cycle_closer", "in1"))

        self.add_components_and_connections(component_list, connection_list)
        # Debug: Print all connections to verify
        print('Connections after setup:')
        for k in self.conn:
            print(k)

    def set_boundary_conditions(self, T_cond=80, T_evap=20, UA_evap=None, area_evap=None, air_inlet_temp=5, air_vol_flow=0.1, air_pressure=1.0, coolant_inlet_temp=35, coolant_pressure=3.0, coolant_mass_flow=0.3):
        p_cond = PSI("P", "Q", 0, "T", 273.15 + T_cond, self.working_fluid) / 1e5
        p_evap = PSI("P", "Q", 1, "T", 273.15 + T_evap, self.working_fluid) / 1e5

        # --- Step 1: Robust starting values (pressures/qualities, unset ttd/kA/Q) ---
        self.conn["evaporator-compressor"].set_attr(p=p_evap, x=1, fluid={self.working_fluid: 1})
        self.conn["condenser-expansionValve"].set_attr(p=p_cond, x=0)
        self.comp["evaporator"].set_attr(kA=None, ttd_l=None, pr1=0.98, pr2=0.98)
        self.comp["condenser"].set_attr(Q=None, ttd_u=None, pr1=0.98, pr2=0.98)
        self.comp["compressor"].set_attr(eta_s=self.compressor_efficiency)
        if self.expansion_device == "expander":
            self.conn["expansionValve-expander"].set_attr(x=0.01)
            self.comp["expander"].set_attr(eta_s=self.expander_efficiency)
        self.conn["evap_air_source-evaporator"].set_attr(fluid={"air": 1}, T=air_inlet_temp, p=air_pressure, v=air_vol_flow)
        self.conn["condenser_coolant_source-condenser"].set_attr(fluid={"water": 1}, T=coolant_inlet_temp, p=coolant_pressure, m=coolant_mass_flow)

        try:
            self.solve()
            print("First solve (robust starting values) succeeded. Switching to design parameters.")
        except Exception as e:
            print("First solve (robust starting values) failed:", e)
            return self

        # --- Step 2: Switch to design parameters (set kA/Q, unset p/flow at sources) ---
        self.comp["evaporator"].set_attr(kA=UA_evap, ttd_l=None)
        self.comp["condenser"].set_attr(Q=-self.Q_out, ttd_u=None)
        self.conn["evaporator-compressor"].set_attr(p=None)
        self.conn["condenser-expansionValve"].set_attr(p=None)
        self.conn["condenser_coolant_source-condenser"].set_attr(T=None, p=None, m=None, fluid=None)
        # (If you set Q on evaporator, unset air source too)
        try:
            self.solve()
            print("Design solve (with kA/Q) succeeded.")
        except Exception as e:
            print("Design solve (with kA/Q) failed:", e)
        return self

    def test_minimal_case(self):
        """Run a minimal case with small Q and large UA to help find good starting values and debug the network."""
        test_Q = 500  # W, small heat flow
        test_UA = 10000  # W/K, very large UA
        test_air_flow = 0.2  # m3/s, reasonable air flow
        test_air_T = 10  # degC
        test_air_p = 1.0  # bar
        test_coolant_T = 30  # degC
        test_coolant_p = 3.0  # bar
        test_coolant_m = 0.3  # kg/s
        self.set_boundary_conditions(
            T_cond=35, T_evap=5, UA_evap=test_UA,
            air_inlet_temp=test_air_T, air_vol_flow=test_air_flow, air_pressure=test_air_p,
            coolant_inlet_temp=test_coolant_T, coolant_pressure=test_coolant_p, coolant_mass_flow=test_coolant_m
        )
        self.comp["condenser"].set_attr(Q=-test_Q)
        print("Running minimal test case with Q=", test_Q, "UA=", test_UA)
        try:
            self.solve()
            print("Minimal case solved! Try using these values as starting points for your main study.")
        except Exception as e:
            print("Minimal case failed:", e)

In [35]:
# Example: Trade-off analysis for different evaporator UA values
UA_values = [500, 700, 900, 1100]  # W/K, example values
COPs = []
for UA in UA_values:
    hp = EvaporatorHeatPumpStudy(working_fluid='R290')
    hp.test_minimal_case()  # Ensure basic setup works
    print(f'\nTesting UA={UA} W/K...')
    hp.set_boundary_conditions(T_cond=50, T_evap=0, UA_evap=UA)
    try:
        hp.solve()
        cop = hp.calculate_cop(consumer='condenser')
        COPs.append(cop)
        print(f'UA={UA} W/K: COP={cop:.2f}')
    except Exception as e:
        COPs.append(np.nan)
        print(f'UA={UA} W/K: Simulation failed ({e})')

You have not provided enough parameters: 12 required, 11 supplied. Aborting calculation!


Connections after setup:
cycle_closer-evaporator
evaporator-compressor
compressor-condenser
condenser-expansionValve
condenser_coolant_source-condenser
condenser-condenser_coolant_sink
evap_air_source-evaporator
evaporator-evap_air_sink
expansionValve-cycle_closer
First solve (robust starting values) failed: You have not provided enough parameters: 12 required, 11 supplied. Aborting calculation!
First solve (robust starting values) failed: You cannot specify two or more values for mass flow in the same linear branch (starting at condenser_coolant_source and ending at condenser_coolant_sink).
Running minimal test case with Q= 500 UA= 10000
Minimal case failed: You cannot specify two or more values for mass flow in the same linear branch (starting at condenser_coolant_source and ending at condenser_coolant_sink).

Testing UA=500 W/K...
First solve (robust starting values) failed: You cannot specify two or more values for mass flow in the same linear branch (starting at condenser_coolant_

NameError: name 'np' is not defined

In [ ]:
# 1. Import libraries and define engineering correlations
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# TESPy imports (assume installed and configured)
from tespy.networks import Network
from tespy.components import Source, Sink, Compressor, SimpleHeatExchanger, Valve, Pump
from tespy.connections import Connection

# Custom subclass for detailed evaporator simulation
from HeatPumpStudy import HeatPumpStudy
from tespy.components import HeatExchanger, Drum, Valve, Source, Sink, Compressor, Pump, CycleCloser
from tespy.connections import Connection
from CoolProp.CoolProp import PropsSI as PSI

# ...other imports as needed...

## 2. Engineering Correlations and Assumptions

- The heat transfer rate Q = UA * ΔT_lm, where U is overall heat transfer coefficient, A is area, ΔT_lm is log mean temperature difference.
- For a given Q (8 kW), increasing A (evaporator size) allows for lower ΔT, improving COP.
- Evaporator cost and weight scale with area, but not always linearly (check real data).
- Compressor selection impacts achievable ΔT and efficiency, especially with expander energy recovery.

In [ ]:
# 3. Define evaporator size, cost, and weight data (example, to be replaced with real data)
evap_data = pd.DataFrame({
    "UA_W_per_K": [500, 700, 900, 1100],  # Example UA values for evaporators
    "Area_m2": [1.5, 2.0, 2.5, 3.0],
    "DeltaT_K": [12, 10, 8, 6],  # Assumed from engineering correlation
    "Price_EUR": [350, 420, 500, 600],
    "Weight_kg": [18, 22, 27, 33]
})

# Run detailed TESPy simulation for each evaporator UA value
COPs = []
for UA in evap_data["UA_W_per_K"]:
    hp = EvaporatorHeatPumpStudy(working_fluid='R290')
    hp.set_boundary_conditions(T_cond=50, T_evap=0, UA_evap=UA, ttd_l=5)
    try:
        hp.solve()
        cop = hp.calculate_cop(consumer='condenser')
        COPs.append(cop)
        print(f'UA={UA} W/K: COP={cop:.2f}')
    except Exception as e:
        COPs.append(np.nan)
        print(f'UA={UA} W/K: Simulation failed ({e})')
evap_data["COP"] = COPs

# Plot price vs area to check linearity
plt.plot(evap_data["Area_m2"], evap_data["Price_EUR"], marker="o")
plt.xlabel("Evaporator Area [m²]")
plt.ylabel("Price [EUR]")
plt.title("Evaporator Price vs Area")
plt.grid()
plt.show()

# Calculate annual operating cost (example: 5000 heating hours, electricity 0.30 EUR/kWh)
annual_hours = 5000
electricity_price = 0.30  # EUR/kWh
Q = 8  # kW
evap_data["Annual_Op_Cost_EUR"] = (Q * annual_hours / evap_data["COP"]) * electricity_price

# Plot trade-off: Manufacturing vs Operating Cost
plt.figure(figsize=(8,5))
plt.plot(evap_data["Price_EUR"], evap_data["Annual_Op_Cost_EUR"], marker="o")
for i, row in evap_data.iterrows():
    plt.annotate(f"UA={row['UA_W_per_K']}", (row["Price_EUR"], row["Annual_Op_Cost_EUR"]))
plt.xlabel("Evaporator Price [EUR]")
plt.ylabel("Annual Operating Cost [EUR]")
plt.title("Trade-off: Manufacturing vs Operating Cost (TESPy Simulation)")
plt.grid()
plt.show()

## 7. Bill of Materials (BOM) for Reference 8 kW Heat Pump

| Component         | Example Model         | Price [EUR] | Weight [kg] | Notes |
|-------------------|----------------------|-------------|-------------|-------|
| Compressor        | Scroll XYZ-8kW       | 400         | 25          | Suitable for 8 kW |
| Evaporator        | See above            | 350-600     | 18-33       | Varies with ΔT    |
| Condenser         | Plate HEX ABC        | 200         | 12          | For 8 kW          |
| Expander          | Custom expander      | 250         | 8           | For energy recovery |
| Fan               | EC Fan 500mm         | 120         | 6           | For air flow      |
| Controller        | PLC Basic            | 80          | 1           |                   |
| ...               | ...                  | ...         | ...         |                   |

*Replace with real data as available. Validate that all components are suitable for 8 kW heating power.*

## 8. Conclusions

- Increasing evaporator size (lower ΔT) improves COP and reduces operating cost, but increases manufacturing cost and weight.
- The expander system helps maintain higher COP at larger ΔT, partially offsetting efficiency losses.
- The optimal design depends on the balance between upfront cost and long-term savings.
- Use real evaporator price/weight data and validate all BOM components for 8 kW suitability.

In [ ]:
# Plot COP vs UA (evaporator size/performance)
plt.figure()
plt.plot(UA_values, COPs, marker='o')
plt.xlabel('Evaporator UA [W/K]')
plt.ylabel('COP')
plt.title('COP vs Evaporator UA')
plt.grid()
plt.show()

This approach allows you to directly study the impact of technical evaporator parameters (UA, area, ttd_l) on heat pump efficiency using a physically accurate TESPy model. You can further extend the class and analysis for more advanced trade-off studies.